# Hiligaynon NER: Deep Error Diagnostics & Confusion Matrix

This notebook generates granular confusion matrices to expose semantic tagging boundaries. It is structured to help manually inspect and isolate misclassifications, heavily centering on the `ORG` vs `LOC` constraint overlap and the linguistic challenges introduced by Taglish (code-switching).

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import sys
import joblib
import itertools
import tempfile
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification, PreTrainedTokenizerFast

# Ensure parent directory is in path to import from 'training' module
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

print('Diagnostic libraries loaded.')

c:\Users\Dallas\Documents\Dallas\3rd Year - 2nd Sem\Natural Lanugage Processing\FINAL PROJECT\HiliTag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Dallas\Documents\Dallas\3rd Year - 2nd Sem\Natural Lanugage Processing\FINAL PROJECT\HiliTag\.venv\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Diagnostic libraries loaded.


c:\Users\Dallas\Documents\Dallas\3rd Year - 2nd Sem\Natural Lanugage Processing\FINAL PROJECT\HiliTag\.venv\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
### 0. Load Converted Verified CoNLL Data

ALLOWED_ENTITIES = {'PERSON', 'ORG', 'LOC', 'EVENT', 'DATETIME', 'MONEY'}

def filter_label(label):
    if label == 'O':
        return label
    # Enforce standard LOC naming
    if 'LOCATION' in label:
        label = label.replace('LOCATION', 'LOC')
    
    # Extract entity type
    parts = label.split('-')
    if len(parts) > 1 and parts[1] in ALLOWED_ENTITIES:
        return label
    return 'O'

def load_conll_data(filepath):
    """Loads sentences and BIOES/BIO tags from a CoNLL-formatted file."""
    sentences = []
    sentence_labels = []
    
    if not os.path.exists(filepath):
        print(f"Warning: CoNLL file {filepath} not found.")
        return [], []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        words, labels = [], []
        for line in f:
            line = line.strip()
            if not line or line.startswith('-DOCSTART-'):
                if words:
                    sentences.append(words)
                    sentence_labels.append(labels)
                    words, labels = [], []
            else:
                splits = line.split()
                if len(splits) >= 2:
                    words.append(splits[0])
                    # Filter the target label directly upon loading
                    labels.append(filter_label(splits[-1]))
        if words:
            sentences.append(words)
            sentence_labels.append(labels)
    
    return sentences, sentence_labels

# Load test, train, and validation splits from converted_verified
converted_dir = os.path.join(os.getcwd(), '..', 'data', 'converted_verified')
print(f"Loading converted CoNLL files from: {converted_dir}\n")

test_sentences, test_labels = load_conll_data(os.path.join(converted_dir, 'dataset_test_final.conll'))
train_sentences, train_labels = load_conll_data(os.path.join(converted_dir, 'dataset_train_final.conll'))
val_sentences, val_labels = load_conll_data(os.path.join(converted_dir, 'dataset_val_final.conll'))

print(f"Test set: {len(test_sentences)} sentences, {sum(len(s) for s in test_sentences)} tokens")
print(f"Train set: {len(train_sentences)} sentences, {sum(len(s) for s in train_sentences)} tokens")
print(f"Val set: {len(val_sentences)} sentences, {sum(len(s) for s in val_sentences)} tokens")
print(f"\nUnique labels in test set: {sorted(set(label for labels in test_labels for label in labels))}")

Loading converted CoNLL files from: c:\Users\Dallas\Documents\Dallas\3rd Year - 2nd Sem\Natural Lanugage Processing\FINAL PROJECT\HiliTag\evaluation\..\data\converted_verified

Test set: 649 sentences, 14766 tokens
Train set: 5200 sentences, 115159 tokens
Val set: 648 sentences, 14392 tokens

Unique labels in test set: ['B-DATETIME', 'B-EVENT', 'B-LOC', 'B-MONEY', 'B-ORG', 'B-PERSON', 'E-DATETIME', 'E-EVENT', 'E-LOC', 'E-MONEY', 'E-ORG', 'E-PERSON', 'I-DATETIME', 'I-EVENT', 'I-LOC', 'I-MONEY', 'I-ORG', 'I-PERSON', 'O', 'S-DATETIME', 'S-EVENT', 'S-LOC', 'S-MONEY', 'S-ORG', 'S-PERSON']


In [3]:
### 1. Generating the Heatmap

In [4]:
def plot_confusion_matrix(y_true, y_pred, classes, title='NER Confusion Matrix', cmap='Reds'):
    """Plots a seaborn heatmap of the classification confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, 
                xticklabels=classes, yticklabels=classes, 
                cbar=False, linewidths=.5)
    
    plt.ylabel('True Labels', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Labels', fontsize=12, fontweight='bold')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Flatten test labels for analysis
y_true_flat = []
for labels in test_labels:
    y_true_flat.extend(labels)

unique_labels = sorted(set(y_true_flat))
print(f"Total test tokens: {len(y_true_flat)}")
print(f"Label distribution:")
for label in unique_labels:
    count = y_true_flat.count(label)
    pct = 100 * count / len(y_true_flat)
    print(f"  {label}: {count} ({pct:.1f}%)")

Total test tokens: 14766
Label distribution:
  B-DATETIME: 78 (0.5%)
  B-EVENT: 90 (0.6%)
  B-LOC: 164 (1.1%)
  B-MONEY: 42 (0.3%)
  B-ORG: 36 (0.2%)
  B-PERSON: 232 (1.6%)
  E-DATETIME: 78 (0.5%)
  E-EVENT: 90 (0.6%)
  E-LOC: 164 (1.1%)
  E-MONEY: 42 (0.3%)
  E-ORG: 36 (0.2%)
  E-PERSON: 232 (1.6%)
  I-DATETIME: 47 (0.3%)
  I-EVENT: 80 (0.5%)
  I-LOC: 92 (0.6%)
  I-MONEY: 20 (0.1%)
  I-ORG: 24 (0.2%)
  I-PERSON: 157 (1.1%)
  O: 12434 (84.2%)
  S-DATETIME: 33 (0.2%)
  S-EVENT: 65 (0.4%)
  S-LOC: 227 (1.5%)
  S-MONEY: 24 (0.2%)
  S-ORG: 29 (0.2%)
  S-PERSON: 250 (1.7%)


### 2. Hunting Down Misclassifications (ORG vs LOC & Taglish)
The function below is designed to filter out exact sentences where a specific true tag is systematically predicted as a designated false tag.

In [6]:
def inspect_errors(sentences, true_labels, pred_labels, true_target='B-ORG', pred_target='B-LOC'):
    """
    Extracts contextual sentences where models confused true_target for pred_target.
    Used to identify systematic misclassifications (e.g., ORG vs LOC, Taglish code-mixing).
    """
    error_instances = []
    
    for sentence_idx, (words, t_tags, p_tags) in enumerate(zip(sentences, true_labels, pred_labels)):
        for word_idx, (word, t_tag, p_tag) in enumerate(zip(words, t_tags, p_tags)):
            if t_tag == true_target and p_tag == pred_target:
                context_start = max(0, word_idx - 2)
                context_end = min(len(words), word_idx + 3)
                context = " ".join(words[context_start:context_end])
                
                error_instances.append({
                    'sentence_idx': sentence_idx,
                    'word_idx': word_idx,
                    'word': word,
                    'context': context,
                    'true_tag': t_tag,
                    'pred_tag': p_tag
                })
                
    return error_instances

print("Error inspection functions ready.\n")

if 'y_pred_xlm' in locals():
    print("=== Analyzing XLM-RoBERTa ORG vs LOC Misclassifications ===")
    org_loc_errors = inspect_errors(test_sentences, test_labels, y_pred_xlm, 'B-ORG', 'B-LOC')
    print(f"Found {len(org_loc_errors)} instances where B-ORG was wrongly predicted as B-LOC.")
    for err in org_loc_errors[:10]:
        print(f"Word: '{err['word']}' | True: {err['true_tag']} -> Pred: {err['pred_tag']}")
        print(f"Context: {err['context']}\n")
elif 'y_pred_crf' in locals():
    print("=== Analyzing CRF ORG vs LOC Misclassifications ===")
    org_loc_errors = inspect_errors(test_sentences, test_labels, y_pred_crf, 'B-ORG', 'B-LOC')
    print(f"Found {len(org_loc_errors)} instances where B-ORG was wrongly predicted as B-LOC.")
    for err in org_loc_errors[:10]:
        print(f"Word: '{err['word']}' | True: {err['true_tag']} -> Pred: {err['pred_tag']}")
        print(f"Context: {err['context']}\n")
else:
    print("Run the model prediction cells above to populate pred_labels.")

Error inspection functions ready.

Run the model prediction cells above to populate pred_labels.


### 3. Comprehensive Performance Metrics (F1-Score Diagram)
Visualizing the F1-scores across all entity classes allows us to immediately identify which specific tags the model struggles with the most.

In [9]:
def plot_top_misclassifications(y_true, y_pred, top_n=10, title="Top Misclassifications"):
    """
    Groups and visualizes the most frequent confusion bounds.
    """
    errors = {}
    for t, p in zip(y_true, y_pred):
        if t != p and t != 'O':  # Track when a true entity was missed
            pair = (t, p)
            errors[pair] = errors.get(pair, 0) + 1
            
    sorted_errors = sorted(errors.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    labels = [f"True: {true}\nPred: {pred}" for (true, pred), count in sorted_errors]
    counts = [count for (true, pred), count in sorted_errors]
    
    plt.figure(figsize=(12, 6))
    sns.barplot(x=counts, y=labels, palette='magma')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Number of Misclassified Tokens', fontsize=12, fontweight='bold')
    plt.ylabel('Error Type (True -> Predicted)', fontsize=12, fontweight='bold')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_normalized_confusion_matrix(y_true, y_pred, classes, title="Normalized Confusion Matrix"):
    """Plots a confusion matrix normalized by true-label counts (rows sum to 1)."""
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    # Avoid division by zero; convert to float and normalize per-row
    with np.errstate(all='ignore'):
        cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='coolwarm', 
                xticklabels=classes, yticklabels=classes, cbar=True, linewidths=.5)
    plt.ylabel('True Labels', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Labels', fontsize=12, fontweight='bold')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

def plot_label_distribution_pie(y_true, classes, title="Label Distribution (Test Set)"):
    counts = [y_true.count(c) for c in classes]
    plt.figure(figsize=(8, 8))
    plt.pie(counts, labels=classes, autopct='%.1f%%', startangle=140, wedgeprops=dict(width=0.4))
    plt.title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_f1_by_label(y_true, y_pred, classes, title="Per-Label F1 Scores"):
    from sklearn.metrics import precision_recall_fscore_support
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(y_true_arr, y_pred_arr, labels=classes, zero_division=0)
    order = np.argsort(f1)[::-1]
    plt.figure(figsize=(12, 6))
    sns.barplot(x=f1[order], y=[classes[i] for i in order], palette='viridis')
    for i, idx in enumerate(order):
        plt.text(f1[idx] + 0.01, i, f"F1={f1[idx]:.2f} (P={precision[idx]:.2f}, R={recall[idx]:.2f})", va='center')
    plt.xlabel('F1 score', fontsize=12, fontweight='bold')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlim(0, 1.0)
    plt.tight_layout()
    plt.show()

# Call the original top misclassifications plot if predictions are available, and add the new diagrams
preds = None
model_label = None
if 'y_pred_flat_xlm' in locals():
    preds = y_pred_flat_xlm
    model_label = 'XLM-RoBERTa'
elif 'y_pred_flat_crf' in locals():
    preds = y_pred_flat_crf
    model_label = 'CRF'

if preds is not None:
    plot_top_misclassifications(y_true_flat, preds, top_n=10, title=f"{model_label}: Top 10 Token Misclassifications")
    # Normalized confusion matrix (row-normalized) to inspect relative confusions
    plot_normalized_confusion_matrix(y_true_flat, preds, unique_labels, title=f"{model_label}: Normalized Confusion Matrix (Test Set)")
    # Per-label F1 scores
    plot_f1_by_label(y_true_flat, preds, unique_labels, title=f"{model_label}: Per-Label F1 Scores")
    # Label distribution pie
    plot_label_distribution_pie(y_true_flat, unique_labels)
else:
    # If no flattened predictions are available, fall back to the existing top misclassifications check
    if 'y_pred_flat_xlm' in locals():
        plot_top_misclassifications(y_true_flat, y_pred_flat_xlm, top_n=10, title="XLM-RoBERTa: Top 10 Token Misclassifications")
    elif 'y_pred_flat_crf' in locals():
        plot_top_misclassifications(y_true_flat, y_pred_flat_crf, top_n=10, title="CRF: Top 10 Token Misclassifications")